# Regime Model

1. **Preprocess** — Compute 6 causal features: ΔDI, slope, ER, z_ATR, BBW, Imbalance
2. **Label** — Generate offline regime labels (TREND / CHOP / TRANSITION)
3. **Train** — Fit classifier (Logistic Regression or MLP) with grid search
4. **Postprocess** — HMM forward-backward decode + transition rules

---

## 0. Setup & Install Dependencies

In [9]:
!pip install -q nautilus_trader scikit-learn plotly pandas numpy

In [10]:
import sys
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
    print("Running in Google Colab")
except ImportError:
    IN_COLAB = False
    print("Running locally")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Running in Google Colab


In [11]:
# ── Set the path to the regime_model_v2 folder ──
# Colab:  PROJECT_ROOT = "/content/drive/MyDrive/regime_model_v2"
# Local:  PROJECT_ROOT = "/path/to/regime_model_v2"

PROJECT_ROOT = "."  # default: notebook is inside regime_model_v2/

if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)
print(f"Working directory: {os.getcwd()}")

Working directory: /content


In [12]:
# Verify imports
from regime_sandbox.constants import (
    FEATURE_COLUMNS, LABEL_STR, REGIME_LABELS,
    TREND, TRANSITION, CHOP,
)
from regime_sandbox.preprocess.config import PreprocessConfig
from regime_sandbox.label.config import RegimeLabelConfig
from regime_sandbox.train.config import TrainConfig
from regime_sandbox.postprocess.config import PostprocessConfig

print("All imports OK")
print(f"Features: {FEATURE_COLUMNS}")
print(f"Labels:   {LABEL_STR}")

ModuleNotFoundError: No module named 'regime_sandbox'

## 1. Configure Paths

Update these to point to your ParquetDataCatalog.

In [ ]:
# ── Data source ──
CATALOG_PATH   = "sample_data/catalog"                       # ParquetDataCatalog root
BAR_TYPE_STR   = "GC.n.0.GLBX-5-MINUTE-LAST-EXTERNAL"       # bar type string
TICK_INSTR     = "GC.n.0.GLBX"                              # instrument ID for ticks
START          = "2020-02-03 00:00:00"
END            = "2020-03-21 00:00:00"

# ── Output directories ──
PREPROCESS_OUT = "output/preprocess"
LABEL_OUT      = "output/label"
TRAIN_OUT      = "output/train"
POSTPROCESS_OUT= "output/postprocess"

# ── Model selection: "logistic" or "mlp" ──
MODEL_TYPE     = "logistic"

print("Configuration set.")

---
## 2. Preprocess — Compute Features

| # | Feature | Formula |
|---|---------|--------|
| 1 | `feat_delta_di` | DI⁺ - DI⁻ |
| 2 | `feat_slope` | (MA_t - MA_{t-k}) / k |
| 3 | `feat_er` | Kaufman Efficiency Ratio |
| 4 | `feat_atr_z` | z-scored ATR |
| 5 | `feat_bbw` | 2·k_B·σ / SMA |
| 6 | `feat_imbalance` | Σ sgn(Δp)·v / Σ v |

In [ ]:
from pathlib import Path
from regime_sandbox.data_loader import load_bars, load_ticks
from regime_sandbox.preprocess.features import compute_features

preprocess_cfg = PreprocessConfig(
    catalog_path=CATALOG_PATH,
    bar_type_str=BAR_TYPE_STR,
    tick_type_str=TICK_INSTR,
    start=START,
    end=END,
    output_dir=PREPROCESS_OUT,
)

# Load bars
bars_df = load_bars(preprocess_cfg.catalog_path, preprocess_cfg.bar_type_str,
                    preprocess_cfg.start, preprocess_cfg.end)

# Load ticks for imbalance
ticks_df = load_ticks(preprocess_cfg.catalog_path, preprocess_cfg.tick_type_str,
                      preprocess_cfg.start, preprocess_cfg.end)

# Compute features
features_df = compute_features(bars_df, preprocess_cfg, ticks_df=ticks_df)

# Save
Path(PREPROCESS_OUT).mkdir(parents=True, exist_ok=True)
features_csv = f"{PREPROCESS_OUT}/features.csv"
features_df[["timestamp"] + FEATURE_COLUMNS].to_csv(features_csv, index=False)

for col in FEATURE_COLUMNS:
    valid = features_df[col].notna().sum()
    print(f"  {col}: {valid:,} valid")

print(f"\nSaved to {features_csv}")
features_df.head()

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np

fig = make_subplots(rows=len(FEATURE_COLUMNS), cols=1, shared_xaxes=True,
                    subplot_titles=FEATURE_COLUMNS, vertical_spacing=0.02)
for i, col in enumerate(FEATURE_COLUMNS):
    v = features_df[col].notna()
    fig.add_trace(go.Scatter(x=features_df.index[v], y=features_df.loc[v, col],
                             mode='lines', name=col, line=dict(width=1)),
                  row=i+1, col=1)
fig.update_layout(height=200*len(FEATURE_COLUMNS), showlegend=False,
                  title="Feature Overview", template="plotly_white")
fig.show()

---
## 3. Label — Generate Regime Labels

- **TREND**: Eff_{t,H} >= 0.15 AND |R^ATR| >= 5.0
- **CHOP**: Eff_{t,H} <= 0.10 AND |R^ATR| <= 3.0
- **TRANSITION**: otherwise

Smoothed with majority-vote + minimum segment enforcement.

In [ ]:
from regime_sandbox.label.labeler import compute_raw_labels
from regime_sandbox.label.smoother import smooth_labels

label_cfg = RegimeLabelConfig(
    catalog_path=CATALOG_PATH,
    bar_type_str=BAR_TYPE_STR,
    instrument_id=TICK_INSTR,
    start=START,
    end=END,
    output_dir=LABEL_OUT,
)

label_df = bars_df.copy()
label_df = compute_raw_labels(label_df, label_cfg)

# Smooth
valid_mask = label_df["raw_label"].notna()
raw = label_df.loc[valid_mask, "raw_label"].values.astype(int)
n_valid = int(valid_mask.sum())

if n_valid == 0:
    print("WARNING: no valid raw labels")
    smoothed = np.array([], dtype=int)
else:
    smoothed = smooth_labels(raw, label_cfg)

label_df["regime_label"] = np.nan
label_df.loc[valid_mask, "regime_label"] = smoothed.astype(float)
label_df["regime_str"] = label_df["regime_label"].map(
    {float(k): v for k, v in LABEL_STR.items()}
)

# Save
Path(LABEL_OUT).mkdir(parents=True, exist_ok=True)
labels_csv = f"{LABEL_OUT}/regime_labels.csv"
label_df[["timestamp", "open", "high", "low", "close", "volume",
          "atr", "efficiency", "abs_r_atr",
          "raw_label", "regime_label", "regime_str"]].to_csv(labels_csv, index=False)

if n_valid > 0:
    for li, ls in LABEL_STR.items():
        count = int((smoothed == li).sum())
        print(f"  {ls}: {count} ({count/n_valid:.1%})")

print(f"\nSaved to {labels_csv}")

In [ ]:
# Visualize labels on price chart
fig = make_subplots(rows=3, cols=1, shared_xaxes=True,
                    row_heights=[0.5, 0.25, 0.25],
                    subplot_titles=["Price + Regime", "Efficiency", "|R/ATR|"])

fig.add_trace(go.Scatter(x=label_df.index, y=label_df["close"],
                         mode="lines", name="Close", line=dict(color="black", width=1)),
              row=1, col=1)

# Background shading
COLOR_MAP = {TREND: "green", TRANSITION: "gold", CHOP: "royalblue"}
valid_labels = label_df["regime_label"].dropna()
if len(valid_labels) > 0:
    changes = valid_labels.ne(valid_labels.shift()).cumsum()
    for seg_id in changes.unique():
        seg = label_df.loc[changes[changes == seg_id].index]
        regime = seg["regime_label"].iloc[0]
        if pd.isna(regime): continue
        fig.add_vrect(x0=seg.index[0], x1=seg.index[-1],
                      fillcolor=COLOR_MAP.get(int(regime), "gray"),
                      opacity=0.15, line_width=0, row=1, col=1)

v = label_df["efficiency"].notna()
fig.add_trace(go.Scatter(x=label_df.index[v], y=label_df.loc[v, "efficiency"],
                         mode="lines", name="Eff", line=dict(width=1)), row=2, col=1)
fig.add_hline(y=label_cfg.eff_trend_min, row=2, col=1, line_dash="dash", line_color="green")
fig.add_hline(y=label_cfg.eff_chop_max, row=2, col=1, line_dash="dash", line_color="blue")

v = label_df["abs_r_atr"].notna()
fig.add_trace(go.Scatter(x=label_df.index[v], y=label_df.loc[v, "abs_r_atr"],
                         mode="lines", name="|R/ATR|", line=dict(width=1)), row=3, col=1)
fig.add_hline(y=label_cfg.abs_return_atr_trend_min, row=3, col=1, line_dash="dash", line_color="green")
fig.add_hline(y=label_cfg.abs_return_atr_chop_max, row=3, col=1, line_dash="dash", line_color="blue")

fig.update_layout(height=800, showlegend=False, template="plotly_white", title="Regime Labels")
fig.show()

---
## 4. Train — Fit Classifier

- **Logistic regression**: `q_t(c) = softmax(W·x + b)`
- **MLP** (single hidden layer): `h = ReLU(W1·x + b1)`, `q_t(c) = softmax(W2·h + b2)`

Selection metric: Macro-F1 on validation split.  
Also saves `hmm_train_labels.npy` for the HMM decode stage.

In [ ]:
from regime_sandbox.train.trainer import train_and_evaluate

train_cfg = TrainConfig(
    labels_csv=labels_csv,
    features_csv=features_csv,
    output_dir=TRAIN_OUT,
    model_type=MODEL_TYPE,
)

report = train_and_evaluate(train_cfg)

In [ ]:
import json
print(json.dumps({
    "model_type": report["model_type"],
    "best_params": report["best_params"],
    "test_accuracy": report["test_accuracy"],
    "test_balanced_accuracy": report["test_balanced_accuracy"],
    "test_macro_f1": report["test_macro_f1"],
    "hmm_train_label_counts": report.get("hmm_train_label_counts"),
}, indent=2, default=str))

In [ ]:
# Confusion matrix
import plotly.figure_factory as ff

cm = np.array(report["test_confusion_matrix"])
class_names = report.get("report_label_order",
                         [LABEL_STR.get(c, str(c)) for c in report["classes"]])

fig = ff.create_annotated_heatmap(
    z=cm, x=class_names, y=class_names,
    colorscale="Blues", showscale=True
)
fig.update_layout(title="Test Confusion Matrix",
                  xaxis_title="Predicted", yaxis_title="Actual",
                  yaxis=dict(autorange="reversed"))
fig.show()

---
## 5. Postprocess — HMM Decode + Transition Rules

1. Load `hmm_train_labels.npy` saved by the trainer
2. Estimate transition matrix A (Laplace smoothed)
3. Forward-backward → posterior γ_t(j)
4. Transition rules: no direct TREND↔CHOP, confirmation bars, min time-in-state

In [ ]:
import pickle
from regime_sandbox.postprocess.hmm_decode import (
    hmm_decode, estimate_transition_matrix,
    STATES, STATE_TO_IDX,
)
from regime_sandbox.postprocess.transition_rules import apply_transition_rules

post_cfg = PostprocessConfig(
    model_dir=TRAIN_OUT,
    features_csv=features_csv,
    labels_csv=labels_csv,
    output_dir=POSTPROCESS_OUT,
)

# Load model + scaler
with open(f"{TRAIN_OUT}/model.pkl", "rb") as f:
    model = pickle.load(f)
with open(f"{TRAIN_OUT}/scaler.pkl", "rb") as f:
    scaler = pickle.load(f)

# Load HMM training labels (saved by trainer)
hmm_labels_path = Path(TRAIN_OUT) / "hmm_train_labels.npy"
if hmm_labels_path.exists():
    train_labels = np.load(hmm_labels_path).astype(int)
    print(f"Loaded {len(train_labels)} HMM training labels")
else:
    print("WARNING: hmm_train_labels.npy not found, using uniform priors")
    train_labels = None

classes = model.classes_.tolist()
print(f"Model classes: {[LABEL_STR.get(c, c) for c in classes]}")

In [ ]:
# Prepare features
feat_df = pd.read_csv(features_csv)
feat_df["timestamp"] = pd.to_datetime(feat_df["timestamp"], utc=True).dt.tz_localize(None)
valid_mask = feat_df[FEATURE_COLUMNS].notna().all(axis=1)
feat_valid = feat_df[valid_mask].reset_index(drop=True)

X = feat_valid[FEATURE_COLUMNS].values
X_scaled = scaler.transform(X)
raw_probs = model.predict_proba(X_scaled)

# Reorder to STATES = [CHOP, TRANSITION, TREND]
emission_probs = np.full((len(raw_probs), len(STATES)), post_cfg.eps, dtype=float)
for i, state in enumerate(STATES):
    if state in classes:
        emission_probs[:, i] = raw_probs[:, classes.index(state)]
emission_probs /= np.maximum(emission_probs.sum(axis=1, keepdims=True), post_cfg.eps)

# HMM decode
gamma, hmm_states = hmm_decode(emission_probs, train_labels,
                                alpha_smooth=post_cfg.alpha_smooth)

# Transition rules
final_states = apply_transition_rules(hmm_states, gamma, post_cfg)

print("\nFinal regime distribution:")
for li, ls in LABEL_STR.items():
    count = (final_states == li).sum()
    print(f"  {ls}: {count} ({count/len(final_states):.1%})")

In [ ]:
# Save results
Path(POSTPROCESS_OUT).mkdir(parents=True, exist_ok=True)

lab_df_ts = pd.read_csv(labels_csv)
lab_df_ts["timestamp"] = pd.to_datetime(lab_df_ts["timestamp"], utc=True).dt.tz_localize(None)
price_merged = pd.merge(feat_valid[["timestamp"]], lab_df_ts[["timestamp", "close"]],
                        on="timestamp", how="left")

result_df = pd.DataFrame({
    "timestamp": feat_valid["timestamp"].values,
    "close": price_merged["close"].values,
    "hmm_state": [LABEL_STR.get(s, "?") for s in hmm_states],
    "final_state": [LABEL_STR.get(s, "?") for s in final_states],
    "gamma_chop": gamma[:, STATE_TO_IDX[CHOP]],
    "gamma_transition": gamma[:, STATE_TO_IDX[TRANSITION]],
    "gamma_trend": gamma[:, STATE_TO_IDX[TREND]],
})
result_df.to_csv(f"{POSTPROCESS_OUT}/final_states.csv", index=False)
print(f"Saved to {POSTPROCESS_OUT}/final_states.csv")
result_df.head(10)

In [ ]:
# ── Final visualization ──

close_vals = price_merged["close"].values
ts = pd.to_datetime(feat_valid["timestamp"].values)

fig = make_subplots(rows=4, cols=1, shared_xaxes=True,
                    row_heights=[0.35, 0.10, 0.25, 0.30],
                    vertical_spacing=0.03,
                    subplot_titles=["Price + Final Regime",
                                   "Final State",
                                   "HMM Posterior γ(j)",
                                   "Transition Matrix A"])

# Row 1: Price + background
fig.add_trace(go.Scatter(x=ts, y=close_vals, mode="lines", name="Close",
                         line=dict(color="black", width=1)), row=1, col=1)
regime_colors = {TREND: "rgba(0,180,0,0.18)", TRANSITION: "rgba(255,200,0,0.15)",
                 CHOP: "rgba(100,100,200,0.18)"}
changes = np.where(np.diff(final_states) != 0)[0] + 1
starts = np.concatenate(([0], changes))
ends = np.concatenate((changes, [len(final_states)]))
for s, e in zip(starts, ends):
    color = regime_colors.get(int(final_states[s]), "rgba(150,150,150,0.08)")
    fig.add_vrect(x0=ts[s], x1=ts[min(e-1, len(ts)-1)],
                  fillcolor=color, line_width=0, row=1, col=1)

# Row 2: State step
fig.add_trace(go.Scatter(x=ts, y=final_states, mode="lines", name="State",
                         line=dict(color="black", width=1),
                         fill="tozeroy", fillcolor="rgba(100,100,200,0.1)"),
              row=2, col=1)

# Row 3: Posteriors
fig.add_trace(go.Scatter(x=ts, y=gamma[:, STATE_TO_IDX[TREND]],
                         mode="lines", name="γ(TREND)",
                         line=dict(color="green", width=1)), row=3, col=1)
fig.add_trace(go.Scatter(x=ts, y=gamma[:, STATE_TO_IDX[CHOP]],
                         mode="lines", name="γ(CHOP)",
                         line=dict(color="blue", width=1)), row=3, col=1)
fig.add_trace(go.Scatter(x=ts, y=gamma[:, STATE_TO_IDX[TRANSITION]],
                         mode="lines", name="γ(TRANS)",
                         line=dict(color="orange", width=1)), row=3, col=1)

# Row 4: Transition matrix heatmap
A = estimate_transition_matrix(train_labels, alpha=post_cfg.alpha_smooth)
state_names = [LABEL_STR[s] for s in STATES]
fig.add_trace(go.Heatmap(
    z=A, x=state_names, y=state_names,
    colorscale="Blues", showscale=True,
    text=np.round(A, 3), texttemplate="%{text}",
), row=4, col=1)

fig.update_layout(height=1200, template="plotly_white",
                  title="Regime Model v2 — Full Pipeline Output", showlegend=True)
fig.update_yaxes(title_text="Price", row=1, col=1)
fig.update_yaxes(title_text="State", row=2, col=1, dtick=1, range=[-0.3, 2.3])
fig.update_yaxes(title_text="Posterior", row=3, col=1, range=[0, 1])
fig.show()

---
## 6. (Optional) Compare Logistic vs MLP

In [ ]:
# Uncomment to train MLP and compare:
#
# mlp_cfg = TrainConfig(
#     labels_csv=labels_csv,
#     features_csv=features_csv,
#     output_dir="output/train_mlp",
#     model_type="mlp",
#     mlp_hidden_widths=(16, 32, 64),
#     mlp_alpha_grid=(1e-4, 1e-3, 1e-2),
# )
# mlp_report = train_and_evaluate(mlp_cfg)
#
# print(f"\nLogistic macro-F1: {report['test_macro_f1']:.4f}")
# print(f"MLP macro-F1:      {mlp_report['test_macro_f1']:.4f}")

---
## 7. (Optional) Tune HMM Parameters

In [ ]:
# Experiment with different HMM and transition rule parameters:
#
# post_tuned = PostprocessConfig(
#     model_dir=TRAIN_OUT,
#     features_csv=features_csv,
#     labels_csv=labels_csv,
#     output_dir="output/postprocess_tuned",
#     alpha_smooth=2.0,
#     trend_confirm_bars=5,
#     chop_confirm_bars=5,
#     t_min=10,
# )
#
# gamma2, hmm2 = hmm_decode(emission_probs, train_labels,
#                            alpha_smooth=post_tuned.alpha_smooth)
# final2 = apply_transition_rules(hmm2, gamma2, post_tuned)
# for li, ls in LABEL_STR.items():
#     print(f"  {ls}: {(final2==li).sum()} ({(final2==li).mean():.1%})")